# TP2 Starter Pack: Image-to-Prompt Inversion

This notebook provides the basic utilities needed for TP2:

- mount Google Drive in Colab / VS Code Colab extension;
- load the TP2 target images;
- extract the fixed render seed from each target filename;
- load the LCM generator `SimianLuo/LCM_Dreamshaper_v7`;
- generate images from prompts using the same settings as the target images;
- save generated images and metadata.

The goal of TP2 is to find prompts that reproduce the target images as closely as possible when rendered with the same LCM setup.


## 1. Install Dependencies

Run this cell first in Colab. Ignore last errors.


In [ ]:
# For local development: run `uv sync` at the project root once.
# Dependencies are pinned in pyproject.toml / uv.lock.
#
# Colab / fresh-runtime users: uncomment the lines below.
# Pillow 12 can break some Diffusers/Transformers imports in Colab Python 3.12
# with errors such as: cannot import name '_Ink' from PIL._typing.
#
# !pip install -q -U "diffusers[torch]" transformers accelerate safetensors matplotlib "pandas<3"
# !pip install -q -U clip-interrogator open-clip-torch
# !pip install -q -U --force-reinstall "Pillow<12"

## 2. Target Paths

Targets live in `tp2-chosen/` (already committed to the repo). Generated
outputs go to `students/outputs/<timestamp>_<identity>/`.

For Colab use, uncomment the Google Drive block at the top of the next cell.

In [ ]:
from pathlib import Path

# --- Colab / Google Drive setup (uncomment in Colab) -----------------------
# import zipfile
# import urllib.request
#
# MOUNT_GOOGLE_DRIVE = True
#
# if MOUNT_GOOGLE_DRIVE:
#     try:
#         from google.colab import drive
#         drive.mount("/content/drive", force_remount=False)
#     except Exception as exc:
#         print("Google Drive mount skipped:", exc)
#
# # Optional online URL for a zip with the target images.
# # Leave empty if targets are already available locally or in Drive.
# TARGETS_ZIP_URL = ""
#
# CONTENT_DIR = Path("/content")
# DRIVE_ROOT = Path("/content/drive/MyDrive")
# DRIVE_PROJECT_DIR = DRIVE_ROOT / "GENAI_TP2"
# LOCAL_PROJECT_DIR = CONTENT_DIR / "GENAI_TP2" if CONTENT_DIR.exists() else Path("students")
#
# if DRIVE_ROOT.exists():
#     DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
#     OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"
# elif CONTENT_DIR.exists():
#     OUTPUT_DIR = CONTENT_DIR / "tp2_outputs"
# else:
#     OUTPUT_DIR = Path("students/outputs")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
# TARGET_DIR_CANDIDATES = [
#     Path("students/tp2-chosen"),
#     Path("tp2-chosen"),
#     Path("/content/tp2-chosen"),
#     Path("/content/tp2_targets"),
#     DRIVE_PROJECT_DIR / "tp2-chosen",
#     Path("/content/drive/MyDrive/tp2-chosen"),
#     Path("/content/drive/MyDrive/tp2_targets"),
# ]
#
# ZIP_CANDIDATES = [
#     Path("students/tp2-chosen.zip"),
#     Path("tp2-chosen.zip"),
#     Path("/content/tp2-chosen.zip"),
#     DRIVE_PROJECT_DIR / "tp2-chosen.zip",
#     Path("/content/drive/MyDrive/tp2-chosen.zip"),
# ]
#
# if TARGETS_ZIP_URL:
#     LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
#     downloaded_zip = LOCAL_PROJECT_DIR / "tp2-chosen.zip"
#     urllib.request.urlretrieve(TARGETS_ZIP_URL, downloaded_zip)
#     ZIP_CANDIDATES.insert(0, downloaded_zip)
#     print("Downloaded targets zip to", downloaded_zip)
#
# if not any(list_target_images(c) for c in TARGET_DIR_CANDIDATES):
#     for zip_path in ZIP_CANDIDATES:
#         if zip_path.exists():
#             extract_dir = Path("/content/tp2-chosen") if CONTENT_DIR.exists() else Path("tp2-chosen")
#             extract_dir.mkdir(parents=True, exist_ok=True)
#             with zipfile.ZipFile(zip_path) as zf:
#                 zf.extractall(extract_dir)
#             print(f"Extracted {zip_path} -> {extract_dir}")
#             break
#
# TARGET_DIR = next(
#     (c for c in TARGET_DIR_CANDIDATES if list_target_images(c)),
#     None,
# )
# if TARGET_DIR is None:
#     raise FileNotFoundError(
#         "No target images found. Put images in MyDrive/GENAI_TP2/tp2-chosen, "
#         "or put tp2-chosen.zip in MyDrive/GENAI_TP2, or set TARGETS_ZIP_URL."
#     )
# --------------------------------------------------------------------------

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def list_target_images(path):
    path = Path(path)
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        return [path]
    if not path.exists():
        return []
    return sorted(p for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

# Local-only paths (comment out if using the Colab block above).
TARGET_DIR = Path("tp2-chosen")
OUTPUT_DIR = Path("students/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

target_images = list_target_images(TARGET_DIR)
if not target_images:
    raise FileNotFoundError(f"No target images found under {TARGET_DIR.resolve()}")

print("Target folder:", TARGET_DIR)
print("Output folder:", OUTPUT_DIR)
print("Number of targets:", len(target_images))
target_images

## 3. Utilities: Seeds, Display, and Output Folders

The target filename encodes the render seed:

- `7836.png` uses seed `7836`;
- `1159_25.png` uses seed `1159`.


In [ ]:
import csv
import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image


def seed_from_filename(path, fallback=2026):
    match = re.match(r"^(\d+)", Path(path).stem)
    return int(match.group(1)) if match else fallback


def safe_stem(path):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in Path(path).stem)


def load_image(path):
    return Image.open(path).convert("RGB")


def create_run_dir(base_dir=OUTPUT_DIR, identity="student_run"):
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base_dir) / f"{timestamp}_{identity}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def write_csv(path, rows):
    rows = list(rows)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        Path(path).write_text("")
        return
    fieldnames = []
    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def show_images(paths, cols=3, title=None):
    paths = list(paths)
    if not paths:
        print("No images to show.")
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]
    for ax in [ax for row in axes for ax in row]:
        ax.axis("off")
    for ax, path in zip([ax for row in axes for ax in row], paths):
        ax.imshow(load_image(path))
        ax.set_title(Path(path).name)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for path in target_images:
    print(Path(path).name, "-> seed", seed_from_filename(path))


In [ ]:
show_images(target_images, cols=3, title="TP2 target images")


## 4. Load the LCM Generator

These are the fixed generation settings used for TP2 targets.

In [ ]:
from dataclasses import dataclass
import torch
from diffusers import DiffusionPipeline


@dataclass(frozen=True)
class LCMConfig:
    model_id: str = "SimianLuo/LCM_Dreamshaper_v7"
    seed: int = 2026  # fallback only; target filenames define the real render seed
    num_inference_steps: int = 8
    guidance_scale: float = 8.0
    lcm_origin_steps: int = 50
    width: int = 768
    height: int = 768


config = LCMConfig()


def default_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = default_device()
print("Using device:", device)


def load_lcm_pipeline(config):
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = DiffusionPipeline.from_pretrained(
        config.model_id,
        torch_dtype=dtype,
        use_safetensors=True,
    )
    if hasattr(pipe, "safety_checker"):
        pipe.safety_checker = None
    pipe.to(device)
    return pipe


pipe = load_lcm_pipeline(config)


## 5. Generate and Save Images

Use `render_prompt_for_target(...)` to render a prompt with the seed encoded in the target filename.

In [ ]:
def render_prompt(prompt, seed, pipe=pipe, config=config):
    generator_device = "cpu" if device == "mps" else device
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        num_inference_steps=config.num_inference_steps,
        guidance_scale=config.guidance_scale,
        lcm_origin_steps=config.lcm_origin_steps,
        width=config.width,
        height=config.height,
        output_type="pil",
        generator=generator,
    ).images[0]
    return image


def render_prompt_for_target(prompt, target_path):
    seed = seed_from_filename(target_path, config.seed)
    return render_prompt(prompt, seed=seed)


def save_generated_image(image, run_dir, target_path, prompt_index=1):
    target_dir = Path(run_dir) / safe_stem(target_path)
    target_dir.mkdir(parents=True, exist_ok=True)
    path = target_dir / f"candidate_{prompt_index:03d}.png"
    image.save(path)
    return path


## CLIP-Interrogator Seed Prompts (Part 1)

First stage of the hybrid VLM + LLM approach. For each target image,
use CLIP Interrogator (BLIP caption + CLIP ViT-L/14 modifier retrieval)
to produce a prompt-shaped seed, then render that seed with the fixed
LCM setup. The LLM refinement loop is Part 2 (not in this notebook yet).

In [ ]:
import warnings

# clip_interrogator still calls deprecated torch.cuda.amp.autocast().
# Silence those FutureWarnings without hiding warnings from our own code.
warnings.filterwarnings("ignore", category=FutureWarning, module=r"clip_interrogator(\..*)?")

from clip_interrogator import Config, Interrogator

ci_config = Config(
    # ViT-L-14-quickgelu uses the QuickGELU activation that the OpenAI
    # checkpoint was trained with -- avoids the open_clip mismatch warning
    # and matches the original OpenAI CLIP behaviour exactly.
    clip_model_name="ViT-L-14-quickgelu/openai",
    caption_model_name="blip-large",
    device=device,                        # reuse the LCM device choice
)
interrogator = Interrogator(ci_config)
print("CLIP Interrogator ready on", device)

In [ ]:
from PIL import Image

seed_prompts = {}   # target filename -> CLIP-Interrogator prompt
for target_path in target_images:
    img = Image.open(target_path).convert("RGB")
    prompt = interrogator.interrogate(img)   # mode="best"
    seed_prompts[target_path.name] = prompt
    print(f"{target_path.name}: {prompt}")

In [ ]:
ci_run_dir = create_run_dir(identity="ci_seeds")
seed_rows = []

for target_path in target_images:
    name = target_path.name
    prompt = seed_prompts[name]
    seed = seed_from_filename(target_path, config.seed)
    rendered = render_prompt(prompt, seed=seed)
    image_path = save_generated_image(rendered, ci_run_dir, target_path, prompt_index=1)
    seed_rows.append({
        "target": str(target_path),
        "target_name": name,
        "render_seed": seed,
        "candidate_index": 1,
        "prompt": prompt,
        "render": str(image_path),
        "source": "clip_interrogator_best",
    })

write_csv(ci_run_dir / "seed_prompts.csv", seed_rows)
(ci_run_dir / "seed_prompts.json").write_text(
    json.dumps(seed_rows, indent=2, ensure_ascii=False)
)
print("Saved seed run to:", ci_run_dir)

In [ ]:
pairs = []
for row in seed_rows:
    pairs.append(row["target"])
    pairs.append(row["render"])
show_images(pairs, cols=2, title="Targets (left) vs CLIP-Interrogator seed renders (right)")